# 16.08 - Diffusion scheduler and denoising-loop practice

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Diffusion schedule and denoising-loop verification.

This third diffusion lesson removes training cost and focuses on the equations used at the scheduler boundary: construct a beta schedule, add known noise, and reverse it with an oracle noise prediction.

## Core Ideas

A beta schedule defines how quickly signal is destroyed. With `alpha_t = 1-beta_t` and cumulative `alpha_bar_t`, a clean sample can be noised directly at any timestep. A reverse sampler combines the current sample with a predicted noise tensor. Using the true noise as an oracle is a powerful unit test: if the schedule indexing and broadcasting are correct, reconstruction error should approach numerical precision.

In [ ]:
import numpy as np
import pandas as pd
import torch

SEED = 16
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Clean Images and Noise

Eight tiny single-channel masks act as images. One fixed noise tensor makes every scheduler comparison reproducible.

In [ ]:
clean_images = torch.zeros((8, 1, 8, 8), dtype=torch.float32)
for index in range(8):
    start = 1 + index % 3
    clean_images[index, 0, start:start + 4, 2:6] = 1.0
clean_images = clean_images * 2.0 - 1.0
oracle_noise = torch.randn_like(clean_images)
print("clean/noise:", clean_images.shape, clean_images.dtype, float(clean_images.min()), float(clean_images.max()))

## Exercise 16-A: Build a linear beta schedule

Keep every schedule tensor on CPU with float32 dtype so later coefficients broadcast predictably.

**Return structure — `make_linear_schedule`:** A dictionary containing `betas`, `alphas`, and `alpha_bars`, each a CPU `torch.float32` tensor of shape `[T]`. `alpha_bars` is the cumulative product of `alphas`.

In [ ]:
def make_linear_schedule(num_steps=20, beta_start=0.0001, beta_end=0.02):
    if num_steps < 2 or not 0.0 < beta_start < beta_end < 1.0:
        raise ValueError("require num_steps >= 2 and 0 < beta_start < beta_end < 1")
    betas = torch.linspace(float(beta_start), float(beta_end), int(num_steps), dtype=torch.float32)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return {"betas": betas, "alphas": alphas, "alpha_bars": alpha_bars}


# Smoke check: inspect the prepared twenty-step schedule.
schedule = make_linear_schedule()
print("schedule endpoints:", schedule["betas"][[0, -1]], schedule["alpha_bars"][[0, -1]])

## Exercise 16-B: Add noise at selected timesteps

Support either one integer timestep for the whole batch or a tensor `[N]` with one timestep per observation.

**Return structure — `add_known_noise`:** A CPU float tensor with the same `[N,C,H,W]` shape and dtype as `clean`. It represents `sqrt(alpha_bar_t)*clean + sqrt(1-alpha_bar_t)*noise`.

In [ ]:
def add_known_noise(clean, noise, timesteps, alpha_bars):
    clean = clean.detach().cpu().to(torch.float32)
    noise = noise.detach().cpu().to(torch.float32)
    alpha_bars = alpha_bars.detach().cpu().to(torch.float32)
    if clean.shape != noise.shape or clean.ndim != 4:
        raise ValueError("clean and noise must share shape [N,C,H,W]")
    if isinstance(timesteps, int):
        timestep_values = torch.full((len(clean),), timesteps, dtype=torch.int64)
    else:
        timestep_values = torch.as_tensor(timesteps, dtype=torch.int64)
    if timestep_values.shape != (len(clean),):
        raise ValueError("timesteps must be an int or shape [N]")
    selected = alpha_bars[timestep_values].reshape(-1, 1, 1, 1)
    return selected.sqrt() * clean + (1.0 - selected).sqrt() * noise


# Smoke check: create the final-timestep noisy batch.
last_timestep = len(schedule["alpha_bars"]) - 1
noisy_images = add_known_noise(clean_images, oracle_noise, last_timestep, schedule["alpha_bars"])
print("noisy batch:", noisy_images.shape, float(noisy_images.mean()))

## Exercise 16-C: Implement one deterministic reverse step

First estimate the clean sample, then move to the previous cumulative-noise level while reusing the predicted noise. At timestep zero, return the clean estimate.

**Return structure — `deterministic_reverse_step`:** A tuple `(previous_sample, clean_estimate)`. Both are detached CPU `torch.float32` tensors with the same `[N,C,H,W]` shape as `current_sample`.

In [ ]:
def deterministic_reverse_step(current_sample, predicted_noise, timestep, alpha_bars):
    current_sample = current_sample.detach().cpu().to(torch.float32)
    predicted_noise = predicted_noise.detach().cpu().to(torch.float32)
    alpha_bars = alpha_bars.detach().cpu().to(torch.float32)
    timestep = int(timestep)
    if current_sample.shape != predicted_noise.shape or timestep < 0 or timestep >= len(alpha_bars):
        raise ValueError("invalid shape or timestep")
    alpha_bar = alpha_bars[timestep]
    clean_estimate = (current_sample - (1.0 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()
    if timestep == 0:
        previous_sample = clean_estimate
    else:
        previous_alpha_bar = alpha_bars[timestep - 1]
        previous_sample = previous_alpha_bar.sqrt() * clean_estimate + (1.0 - previous_alpha_bar).sqrt() * predicted_noise
    return previous_sample.to(torch.float32), clean_estimate.to(torch.float32)


# Smoke check: reverse the final schedule step with oracle noise.
previous_images, clean_estimate = deterministic_reverse_step(noisy_images, oracle_noise, last_timestep, schedule["alpha_bars"])
print("reverse outputs:", previous_images.shape, clean_estimate.shape)

## Exercise 16-D: Run the complete oracle denoising loop

Store the initial noisy batch and every subsequent reverse state so indexing errors are visible.

**Return structure — `oracle_denoising_loop`:** A tuple `(reconstruction, trajectory)`. `reconstruction` has shape `[N,C,H,W]`; `trajectory` has shape `[T+1,N,C,H,W]`; both are detached CPU float32 tensors.

In [ ]:
def oracle_denoising_loop(final_noisy_sample, known_noise, alpha_bars):
    current = final_noisy_sample.detach().cpu().to(torch.float32)
    states = [current.clone()]
    for timestep in range(len(alpha_bars) - 1, -1, -1):
        current, clean_estimate = deterministic_reverse_step(current, known_noise, timestep, alpha_bars)
        states.append(current.clone())
    return current, torch.stack(states)


# Smoke check: reverse all twenty steps.
reconstructed_images, denoising_trajectory = oracle_denoising_loop(noisy_images, oracle_noise, schedule["alpha_bars"])
print("trajectory:", denoising_trajectory.shape)

## Exercise 16-E: Summarize reconstruction evidence

Report MSE at the initial noisy state, the midpoint, and the final reconstruction.

**Return structure — `trajectory_error_table`:** A three-row `pandas.DataFrame` with integer `state_index` and float `mse` columns.

In [ ]:
def trajectory_error_table(trajectory, clean):
    trajectory = trajectory.detach().cpu().to(torch.float32)
    clean = clean.detach().cpu().to(torch.float32)
    indices = [0, len(trajectory) // 2, len(trajectory) - 1]
    return pd.DataFrame([{"state_index": int(index), "mse": float(torch.mean((trajectory[index] - clean) ** 2))} for index in indices])


# Smoke check and full prepared-batch evidence.
diffusion_evidence = trajectory_error_table(denoising_trajectory, clean_images)
print(diffusion_evidence.to_string(index=False))

## Test Cases

**Return structure — `run_day16_tests`:** Returns `None`; assertions and `Day 16 tests passed` communicate success.

In [ ]:
def run_day16_tests():
    assert set(schedule) == {"betas", "alphas", "alpha_bars"}
    assert all(values.shape == (20,) and values.dtype == torch.float32 for values in schedule.values())
    assert bool(torch.all(schedule["alpha_bars"][1:] < schedule["alpha_bars"][:-1]))
    assert noisy_images.shape == clean_images.shape
    assert previous_images.shape == clean_images.shape and clean_estimate.shape == clean_images.shape
    assert denoising_trajectory.shape == (21, 8, 1, 8, 8)
    assert torch.mean((reconstructed_images - clean_images) ** 2).item() < 1e-10
    assert list(diffusion_evidence.columns) == ["state_index", "mse"] and len(diffusion_evidence) == 3
    assert float(diffusion_evidence.iloc[-1]["mse"]) < 1e-10
    print("Day 16 tests passed")


run_day16_tests()

## Day 16 Checklist

- [ ] Explain beta, alpha, and cumulative alpha-bar.
- [ ] Broadcast timestep coefficients across `[N,C,H,W]`.
- [ ] Recover a clean estimate from known noise.
- [ ] Verify the complete reverse-loop trajectory.
- [ ] Run the test cases.